In [ ]:
! pip install requests
! pip install langchain_community
! pip install -U langchain-huggingface
! pip install pinecone-client 
! pip install transformers
! pip install sentence_transformers
! pip install llama_index  
! pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cpu
! pip install -U langchain-huggingface
! pip install python-dotenv


In [ ]:
from flask import Flask,render_template, request, jsonify,redirect,url_for
from requests.auth import HTTPBasicAuth
import requests
import json
import pinecone
import couchdb
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.llms import HuggingFaceHub
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from InstructorEmbedding import INSTRUCTOR
#from langchain import HuggingFaceHub
import time
from InstructorEmbedding import INSTRUCTOR
from pinecone import Pinecone, ServerlessSpec
#from langchain.chains import retrieval_qaWithsourceChain
#from langchain.chains.question_answering import load_qa_chain

#from some_module import HuggingFaceInstructEmbeddings 

In [ ]:
app = Flask(__name__)


In [ ]:
# CouchDB connection parameters
COUCHDB_URL = 'https://192.168.57.185:5984'
COUCHDB_USERNAME = 'd_couchdb'
COUCHDB_PASSWORD = 'Welcome#2'
DATABASE_NAME = 'tamil_datalinkpro'

In [ ]:
# Disable SSL verification for the development environment
requests.packages.urllib3.disable_warnings()


In [ ]:
def connect_to_couchdb(db_name, couchdb_url, username, password):
    # Connect to the CouchDB server
    server = couchdb.Server(couchdb_url)
    
    # Authentication if required
    if username and password:
        server.resource.credentials = (username, password)
    
    # Connect to the specific database
    if db_name in server:
        db = server[db_name]
        print(f"Connected to database '{db_name}'")
    else:
        db = server.create(db_name)
        print(f"Database '{db_name}' not found. Created a new database with the same name.")
    
    return db

# Connect to CouchDB server
db = connect_to_couchdb(DATABASE_NAME, COUCHDB_URL, COUCHDB_USERNAME,COUCHDB_PASSWORD)



In [ ]:
def create_or_access_database(url, username, password, database_name):
    try:
        response = requests.put(f"{url}/{database_name}", auth=HTTPBasicAuth(username, password), verify=False)
        if response.status_code == 201:
            print(f"Database {database_name} created successfully.")
        elif response.status_code == 412:
            print(f"Database {database_name} already exists.")
        else:
            print(f"Unexpected response code: {response.status_code}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error creating or accessing the database: {e}")
        return False

if not create_or_access_database(COUCHDB_URL, COUCHDB_USERNAME, COUCHDB_PASSWORD, DATABASE_NAME):
    exit(1)

In [ ]:
def listen_to_changes(db):
    while True:
        # Listen to the changes feed
        for change in db.changes(feed='continuous', include_docs=True, heartbeat=True):
            doc_id = change.get('id')
            doc = change.get('doc')
            
            if doc_id and doc:
                print(f"Document {doc_id} updated and returned")
                yield doc  # Yield the document to the calling code

for updated_doc in listen_to_changes(db):
    # Here, updated_doc contains the latest document change
    # You can process it as needed
    print("Processing document:", updated_doc)

In [ ]:
def retrieve_all_documents(url, username, password, database_name):
    try:
        all_docs_url = f"{url}/{database_name}/_all_docs?include_docs=true"
        auth = HTTPBasicAuth(username, password)
        response = requests.get(all_docs_url, auth=auth, verify=False)
        response.raise_for_status()
        docs = response.json()
        
        # Convert the entire JSON data to a single string
        document = json.dumps(docs, indent=4)

        # Initialize an empty list to store the concatenated text
        all_texts = []

        # Extract documents from the response
        documents = docs.get('rows', [])

        # Check if documents is a list (i.e., multiple documents)
        if isinstance(documents, list):
            for doc in documents:
                doc_data = doc.get('doc', {})
                # Convert each document to a string format
                if isinstance(doc_data, dict):
                    text = (
                        f"Employee ID: {str(doc_data.get('employee_id', 'N/A'))}\n"
                        f"Name: {doc_data.get('name', 'N/A')}\n"
                        f"Gender: {doc_data.get('gender', 'N/A')}\n"
                        f"Date of Birth: {str(doc_data.get('date_of_birth', 'N/A'))}\n"
                        f"Marital Status: {doc_data.get('marital_status', 'N/A')}\n"
                        f"Nationality: {doc_data.get('nationality', 'N/A')}\n"
                        f"Address: {doc_data.get('address', 'N/A')}\n"
                        f"Contact Number: {doc_data.get('contact_number', 'N/A')}\n"
                        f"Email: {doc_data.get('email', 'N/A')}\n"
                        f"Department: {doc_data.get('department', 'N/A')}\n"
                        f"Job Title: {doc_data.get('job_title', 'N/A')}\n"
                        f"Date of Hire: {str(doc_data.get('date_of_hire', 'N/A'))}\n"
                        f"Employment Type: {doc_data.get('employment_type', 'N/A')}\n"
                        f"Work Location: {doc_data.get('work_location', 'N/A')}\n"
                        f"Supervisor: {doc_data.get('supervisor', 'N/A')}\n"
                        f"Work Authorization: {doc_data.get('work_authorization', 'N/A')}\n"
                        f"Background Check: {doc_data.get('background_check', 'N/A')}\n"
                        f"Employment Agreement: {doc_data.get('employment_agreement', 'N/A')}\n"
                        f"Education: {doc_data.get('education', 'N/A')}\n"
                        f"Certifications: {', '.join(doc_data.get('certifications', []))}\n"
                        f"Previous Experience: {', '.join(doc_data.get('previous_experience', []))}\n"
                        f"References: {', '.join(doc_data.get('references', []))}\n"
                        f"Salary: ${doc_data.get('salary', 'N/A')}\n"
                        f"Bonus: ${doc_data.get('bonus', 'N/A')}\n"
                        f"Commission: ${doc_data.get('commission', 'N/A')}\n"
                        f"Total Working Days: {doc_data.get('total_working_days', 'N/A')}\n"
                        f"Total Worked Days: {doc_data.get('total_worked_days', 'N/A')}\n"
                        f"Leaves Taken: {', '.join([f'{month}: {len(days)} days' for month, days in doc_data.get('leaves_taken', {}).items()])}\n"
                        f"---\n"
                    )
                    all_texts.append(text)

        # Combine all document strings into a single string
        combined_text = '\n'.join(all_texts)

        return combined_text

    except requests.exceptions.RequestException as e:
        print(f"Error retrieving documents: {e}")
        return ""



# retrieval
document = retrieve_all_documents(COUCHDB_URL, COUCHDB_USERNAME, COUCHDB_PASSWORD, DATABASE_NAME)

In [ ]:
def get_text_chunk(document):
    text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=200, length_function=len)
    text_chunks = text_splitter.split_text(document)
    return text_chunks

#converting the JSON file into single string ,converting the single string into text chuncks
text_chunks = get_text_chunk(document)

In [ ]:

# Set up the embeddings
embeddings = HuggingFaceInstructEmbeddings(model_name='hkunlp/instructor-xl')
embeddings

In [ ]:
def get_vectorstore(text_chunks):
   
   # Initialize Pinecone with API key and environment"
    index_name = "llm23"
    # Initialize the Pinecone index
    index = pinecone.Index(index_name="llm23",api_key="PINECONE_API_KEY",host="https://llm23-g9ak0ib.svc.aped-4627-b74a.pinecone.io")
    
   # Convert text chunks to embeddings and flatten the list
    vector_embeddings = [embeddings.embed_documents(text)[0] for text in text_chunks] # Flatten the list of lists

    # Create upsert payload, use string ids
    upsert_payload = [(str(i), vector_embeddings[i]) for i in range(len(text_chunks))]
    print(upsert_payload)

    # Upsert (insert) the vectors into the Pinecone index
    index.upsert(vectors=upsert_payload)
    return index

#Embedding the chunks of data into vector database
vectorstore = get_vectorstore(text_chunks)

In [ ]:
def get_conversation_chain(vectorstore):
    llm_pipeline = HuggingFaceHub(repo_id="google/flan-t5-xxl", model_kwargs={"temperature": 0.5, "max_length": 512})
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm_pipeline, retriever=vectorstore.as_retriever(), memory=memory)
    return conversation_chain

#create conversation chain
conversation = get_conversation_chain(vectorstore)


In [ ]:
##cosine similarity retreive Results from VectorDB
def retreive_results(query,k=2):
  matching_results=vectorstore.similarity_search(query,k=k)
  return matching_results

In [ ]:
from langchain.llms import HuggingFaceHub
from langchain.chains.question_answering